# BenignIDS — Setup & Feature Engineering

**Scope:** Sections 0.1–0.4 (setup, load, target, splits) and 2.1 (payload TF‑IDF).

Run this notebook top‑to‑bottom before 03–10.

## Section 0.1 — Sanity Config, Early Imports & Canonical Lock

In [1]:
# =====================================================
# 0.1 — Config (LABEL_MAP = single source of truth)
# =====================================================
BASE_LABEL_MAP = {
    # negatives
    "benign": 0, "normal": 0, "noattack": 0, "false": 0, "neg": 0, "negative": 0,
    # positives
    "malicious": 1, "attack": 1, "true": 1, "pos": 1, "positive": 1,
}
ANALYSIS_IS_ATTACK = int(globals().get("ANALYSIS_IS_ATTACK", 1))  # 1=malicious, 0=benign
EXTRA_LABEL_MAP    = dict(globals().get("EXTRA_LABEL_MAP", {"analysis": ANALYSIS_IS_ATTACK}))
LABEL_MAP          = {**BASE_LABEL_MAP, **EXTRA_LABEL_MAP}

OUT_ROOT     = globals().get("OUT_ROOT", "out")
SPLITS_DIR   = globals().get("SPLITS_DIR", f"{OUT_ROOT}/splits")
TARGET_COL   = globals().get("TARGET_COL", "label")
DATA_PATH    = globals().get("DATA_PATH", None)  # CSV with features (+ payload and/or labels)
RANDOM_STATE = int(globals().get("RANDOM_STATE", 42))

print("[0.1] OUT_ROOT   :", OUT_ROOT)
print("[0.1] SPLITS_DIR :", SPLITS_DIR)
print("[0.1] DATA_PATH  :", DATA_PATH)
print("[0.1] TARGET_COL :", TARGET_COL)
print("[0.1] LABEL_MAP  :", {k: LABEL_MAP[k] for k in list(LABEL_MAP)[:8]}, "...")


[0.1] OUT_ROOT   : out
[0.1] SPLITS_DIR : out/splits
[0.1] DATA_PATH  : None
[0.1] TARGET_COL : label
[0.1] LABEL_MAP  : {'benign': 0, 'normal': 0, 'noattack': 0, 'false': 0, 'neg': 0, 'negative': 0, 'malicious': 1, 'attack': 1} ...


## Section 0.2 — Load Data (features + optional labels merge)

In [2]:
# =====================================================
# 0.2 — Load & Basic Hygiene
# =====================================================
import os, numpy as np, pandas as pd
if DATA_PATH is None:
    DATA_PATH = "archive/Payload_data_UNSW.csv"
assert DATA_PATH and os.path.exists(DATA_PATH), f"[0.2] Set DATA_PATH to your CSV. Current: {DATA_PATH}"
df = pd.read_csv(DATA_PATH)
print("[0.2] raw shape:", df.shape)

# Stable row id for cross-notebook alignment
if "_ROW_ID" not in df.columns:
    df = df.copy()
    df["_ROW_ID"] = np.arange(len(df))


[0.2] raw shape: (79881, 1505)


## Section 0.2d — Labels Forensics & Manual Override

In [3]:
# =====================================================
# 0.2 — Load & Basic Hygiene
# =====================================================
import os, numpy as np, pandas as pd
assert DATA_PATH and os.path.exists(DATA_PATH), "[0.2] Set DATA_PATH to your CSV."
df = pd.read_csv(DATA_PATH)
print("[0.2] raw shape:", df.shape)

# Stable row id for cross-notebook alignment
if "_ROW_ID" not in df.columns:
    df = df.copy()
    df["_ROW_ID"] = np.arange(len(df))


[0.2] raw shape: (79881, 1505)


In [4]:
# ======================================================
# Section 0.2e — Label Sanitiser (mixed numeric + strings)
#   • Maps known strings (incl. "analysis") → {0,1}
#   • Handles mixed dtypes robustly
#   • Enforces a policy for any remaining unmapped tokens
# ======================================================
print(">>> Section 0.2e — Label Sanitiser: start")

import pandas as pd

assert "label" in df.columns, "[0.2e] df has no 'label' column. Run 0.2d first."

# 1) Mapping dictionary (base + your extra)
BASE_LABEL_MAP = {
    # negatives
    "benign":0, "normal":0, "noattack":0, "false":0, "neg":0, "negative":0,
    # positives
    "malicious":1, "attack":1, "true":1, "pos":1, "positive":1,
}
ANALYSIS_IS_ATTACK = int(globals().get("ANALYSIS_IS_ATTACK", 1))  # set to 0 if "analysis" = benign
EXTRA_LABEL_MAP = dict(globals().get("EXTRA_LABEL_MAP", {"analysis": ANALYSIS_IS_ATTACK}))
LABEL_MAP = {**BASE_LABEL_MAP, **EXTRA_LABEL_MAP}

# 2) Convert numeric-like values; keep strings for mapping
raw = df["label"]
num = pd.to_numeric(raw, errors="coerce")  # numeric values become 0/1/NaN
need_map = num.isna()

# 3) Map strings where numeric failed
mapped = (
    raw[need_map]
    .astype(str).str.strip().str.lower()
    .map(LABEL_MAP)
)

# 4) Combine numeric + mapped strings
y = num.copy()
y.loc[need_map] = mapped

# 5) Handle any still-unmapped tokens via a clear policy
UNMAPPED_POLICY = globals().get("UNMAPPED_POLICY", "drop")
# Options: "drop" (default), "assign_0", "assign_1", "error"

unmapped_mask = y.isna()
if unmapped_mask.any():
    # diagnostics
    tokens = (
        raw[unmapped_mask]
        .astype(str).str.strip().str.lower()
        .value_counts()
    )
    print(f"[0.2e][diag] Unmapped tokens (top 10):\n{tokens.head(10)}")

    if UNMAPPED_POLICY == "assign_0":
        y.loc[unmapped_mask] = 0
        print(f"[0.2e] Assigned {unmapped_mask.sum()} unmapped tokens to 0 (benign).")
    elif UNMAPPED_POLICY == "assign_1":
        y.loc[unmapped_mask] = 1
        print(f"[0.2e] Assigned {unmapped_mask.sum()} unmapped tokens to 1 (attack).")
    elif UNMAPPED_POLICY == "drop":
        # Drop rows with unmapped labels
        kept = ~unmapped_mask
        dropped = int(unmapped_mask.sum())
        df = df.loc[kept].reset_index(drop=True)
        y = y.loc[kept].reset_index(drop=True)
        print(f"[0.2e] Dropped {dropped} rows with unmapped labels. New df shape={df.shape}.")
    else:  # "error"
        raise ValueError(
            "[0.2e] Unmapped label tokens remain and UNMAPPED_POLICY='error'. "
            "Set UNMAPPED_POLICY to 'drop' | 'assign_0' | 'assign_1', "
            "or extend EXTRA_LABEL_MAP."
        )

# 6) Finalise
y = y.astype(int)
df["label"] = y
print(f"[0.2e] Label sanitised — counts: {y.value_counts().to_dict()}")
print(">>> Section 0.2e — Label Sanitiser: complete")


>>> Section 0.2e — Label Sanitiser: start
[0.2e][diag] Unmapped tokens (top 10):
label
generic           17580
exploits          13992
fuzzers           12722
reconnaissance     7562
dos                3397
backdoor           1239
shellcode          1088
worms                93
Name: count, dtype: int64


[0.2e] Dropped 57673 rows with unmapped labels. New df shape=(22208, 1506).
[0.2e] Label sanitised — counts: {0: 21000, 1: 1208}
>>> Section 0.2e — Label Sanitiser: complete


## Section 0.3 — Target Audit & Resolution

In [5]:
# =====================================================
# 0.3 — Resolve binary target (0/1)
# =====================================================
cand = next((c for c in ("label","label_str","attack_cat") if c in df.columns), None)
assert cand is not None, "[0.3] Need one of: label / label_str / attack_cat"

if cand == "label":
    y = pd.to_numeric(df[cand], errors="coerce")
    assert set(pd.unique(y.dropna())) <= {0,1}, "[0.3] 'label' exists but isn’t binary 0/1"
elif cand == "label_str":
    y = df[cand].astype(str).str.strip().str.lower().map(LABEL_MAP)
elif cand == "attack_cat":
    s = df[cand].astype(str).str.strip().str.lower()
    y = (s != "benign").astype(int)

y = y.fillna(1).astype(int)
X = df.drop(columns=[cand])
TARGET_COL = "label"
print("[0.3] target counts:", dict(zip(*np.unique(y, return_counts=True))))


[0.3] target counts: {0: 21000, 1: 1208}


In [6]:
# =====================================================
# 0.4 — Split & Persist (parquet + indices + manifest)
# =====================================================
from pathlib import Path
import json, uuid, time
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split

OUT_ROOT   = Path(OUT_ROOT);  OUT_ROOT.mkdir(parents=True, exist_ok=True)
SPLITS_DIR = Path(SPLITS_DIR); SPLITS_DIR.mkdir(parents=True, exist_ok=True)

X_train, X_tmp, y_train, y_tmp = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE
)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.5, stratify=y_tmp, random_state=RANDOM_STATE
)

def _ids_from_frame(frame):
    return frame["_ROW_ID"].to_numpy() if "_ROW_ID" in frame.columns else np.arange(frame.shape[0])

# --- persist split indices (for external alignment)
np.save(SPLITS_DIR/"train_idx.npy", _ids_from_frame(X_train))
np.save(SPLITS_DIR/"val_idx.npy",   _ids_from_frame(X_val))
np.save(SPLITS_DIR/"test_idx.npy",  _ids_from_frame(X_test))

# --- persist dataframes
X_train.to_parquet(SPLITS_DIR/"X_train.parquet", index=False)
X_val.to_parquet(  SPLITS_DIR/"X_val.parquet",   index=False)
X_test.to_parquet( SPLITS_DIR/"X_test.parquet",  index=False)
pd.DataFrame({"label": y_train}).to_parquet(SPLITS_DIR/"y_train.parquet", index=False)
pd.DataFrame({"label": y_val}).to_parquet(  SPLITS_DIR/"y_val.parquet",   index=False)
pd.DataFrame({"label": y_test}).to_parquet( SPLITS_DIR/"y_test.parquet",  index=False)

# --- JSON-safe counts (cast NumPy scalars to Python ints)
def _counts(arr):
    u, c = np.unique(np.asarray(arr), return_counts=True)
    return {int(k): int(v) for k, v in zip(u.tolist(), c.tolist())}

SPLIT_ID = str(uuid.uuid4())
manifest = {
    "split_id": SPLIT_ID,
    "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "target_col": "label",
    "counts": {
        "train": _counts(y_train),
        "val":   _counts(y_val),
        "test":  _counts(y_test),
    },
    "source": str(DATA_PATH),
    "schema_version": 1,
}

# --- write manifest
(SPLITS_DIR/"manifest.json").write_text(json.dumps(manifest, indent=2))
print(f"[0.4] manifest → {SPLITS_DIR/'manifest.json'}  split_id={SPLIT_ID}")


[0.4] manifest → out/splits/manifest.json  split_id=863c8586-b93c-47ba-90ea-f7ea2497ae5c


In [7]:
import pandas as pd, numpy as np, os
SPLITS_DIR = "out/splits"
ytr = pd.read_parquet(f"{SPLITS_DIR}/y_train.parquet").iloc[:,0].to_numpy()
yva = pd.read_parquet(f"{SPLITS_DIR}/y_val.parquet").iloc[:,0].to_numpy()
print("train:", dict(zip(*np.unique(ytr, return_counts=True))))
print("val  :", dict(zip(*np.unique(yva, return_counts=True))))


train: {0: 15750, 1: 906}
val  : {0: 2625, 1: 151}


## Section 2.1 — Payload Sequence (TF‑IDF)

In [8]:
# ======================================================
# Section 2.1 — Payload Sequence (TF‑IDF)
# ======================================================
print(">>> Section 2.1 — Payload Sequence (TF-IDF): start")

import re
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer

VEC_PATH = Path(OUT_ROOT) / "payload" / "vectorizer.joblib"
VEC_PATH.parent.mkdir(parents=True, exist_ok=True)

if "payload" not in df.columns:
    byte_cols = [c for c in df.columns if re.match(r"^payload_byte_\d+$", c)]
    assert byte_cols, "[2.1] No payload_byte_* columns found."
    df["payload"] = df[byte_cols].astype(str).agg(" ".join, axis=1)
    print(f"[2.1] Folded {len(byte_cols)} byte columns → 'payload'.")

vectorizer = TfidfVectorizer(
    max_features=20000,
    token_pattern=r"\b\d+\b"
)
vectorizer.fit(df["payload"])

joblib.dump(vectorizer, VEC_PATH)
globals()["VEC_PATH"] = VEC_PATH
globals()["vectorizer"] = vectorizer
print(f"[2.1] Saved TF-IDF vectoriser → {VEC_PATH}")
print(">>> Section 2.1 — Payload Sequence (TF-IDF): complete")

>>> Section 2.1 — Payload Sequence (TF-IDF): start


[2.1] Folded 1500 byte columns → 'payload'.


[2.1] Saved TF-IDF vectoriser → out/payload/vectorizer.joblib
>>> Section 2.1 — Payload Sequence (TF-IDF): complete
